# AI Visual Quality Inspection: Baseline Model Training (Milestone 2)

In this notebook, we orchestrate the experiment to train a baseline deep learning model (ResNet18) for classifying casting components as defective or acceptable.

## 1. Import Required Libraries

In [ ]:
import os
import sys
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import seaborn as sns

# Ensure project root is in sys.path to import our custom modules
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

from app.ml.data_loader import get_image_paths_and_labels
from app.ml.preprocessing import split_dataset
from app.ml.model import get_defect_detection_model
from app.ml.train import train_model

## 2. Load Dataset Paths and Labels
We reuse the data loader implemented in Milestone 1 to discover the dataset and split it.

In [ ]:
DATA_DIR = os.path.join(project_root, 'data', 'raw')

# Load image paths and labels
image_paths, labels, idx_to_class = get_image_paths_and_labels(DATA_DIR)
class_names = [idx_to_class[i] for i in range(len(idx_to_class))]

# Split the dataset into Train, Val, and Test sets (70% / 15% / 15%)
X_train, X_val, X_test, y_train, y_val, y_test = split_dataset(
    image_paths, labels, test_size=0.15, val_size=0.15
)

print(f"Training samples: {len(X_train)}")
print(f"Validation samples: {len(X_val)}")
print(f"Test samples: {len(X_test)}")

## 3. Create PyTorch Dataset Class

We need a custom PyTorch Dataset to load the images on the fly and apply torchvision transforms. This prevents loading the entire dataset into RAM at once.

In [ ]:
class CastingDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform
        
    def __len__(self):
        return len(self.image_paths)
        
    def __getitem__(self, idx):
        # PIL is used to read images as it interfaces perfectly with torchvision transforms
        # We convert to RGB because ResNet18 expects 3-channel input
        image = Image.open(self.image_paths[idx]).convert('RGB')
        label = self.labels[idx]
        
        if self.transform:
            image = self.transform(image)
            
        return image, label

## 4. Define Transforms and Create DataLoaders

We standardize the images to 224x224 (optimal for ResNet), convert to PyTorch tensors, and apply ImageNet normalization to match the statistics of the dataset the model was originally trained on.

In [ ]:
# Standard transformations for ResNet architecture
data_transforms = transforms.Compose([
    transforms.Resize((224, 224)),  # Resize to 224x224 required by standard ResNet
    transforms.ToTensor(),          # Convert to tensor and scale pixels [0.0, 1.0]
    transforms.Normalize(           # Normalize using ImageNet mean and std
        mean=[0.485, 0.456, 0.406], 
        std=[0.229, 0.224, 0.225]
    )
])

# Instantiate the datasets
train_dataset = CastingDataset(X_train, y_train, transform=data_transforms)
val_dataset = CastingDataset(X_val, y_val, transform=data_transforms)
test_dataset = CastingDataset(X_test, y_test, transform=data_transforms)

BATCH_SIZE = 32

# Create DataLoaders to handle batching and shuffling
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

## 5. Initialize the Model

We initialize a pre-trained ResNet18 model configured for 2-class binary output.

In [ ]:
model = get_defect_detection_model(num_classes=len(idx_to_class), pretrained=True)

## 6. Train the Model

Execute the training pipeline. The pipeline will automatically save the best model weights based on validation accuracy to `models/defect_detector_resnet18.pth`.

In [ ]:
MODELS_DIR = os.path.join(project_root, 'models')

# For this baseline, we train for 5 epochs. In production, this might be higher with early stopping.
EPOCHS = 5

# Run training (uncomment the next line to actually train the model!)
# best_model, history = train_model(
#     model=model, 
#     train_loader=train_loader, 
#     val_loader=val_loader, 
#     epochs=EPOCHS,
#     learning_rate=1e-4,  # Lower learning rate is better for fine-tuning
#     save_dir=MODELS_DIR
# )

## 7. Plot Training Curves

Visualizing the loss during training helps diagnose overfitting or underfitting issues.

In [ ]:
def plot_training_history(history):
    epochs = range(1, len(history['train_loss']) + 1)
    
    plt.figure(figsize=(12, 5))
    
    # Plot Losses
    plt.subplot(1, 2, 1)
    plt.plot(epochs, history['train_loss'], 'b-', label='Training Loss')
    plt.plot(epochs, history['val_loss'], 'r-', label='Validation Loss')
    plt.title('Training and Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    
    # Plot Validation Accuracy
    plt.subplot(1, 2, 2)
    plt.plot(epochs, history['val_acc'], 'g-', label='Validation Accuracy')
    plt.title('Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    
    plt.tight_layout()
    plt.show()

# Uncomment after training to visualize results:
# plot_training_history(history)

## 8. Evaluate on Test Dataset

We evaluate the best model on the holdout test set to get a final unbiased estimate of its real-world performance.

In [ ]:
def evaluate_model(model, test_loader, class_names):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    model.eval()
    
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
            
    # Calculate accuracy
    acc = accuracy_score(all_labels, all_preds)
    print(f"Test Accuracy: {acc * 100:.2f}%\n")
    
    # Classification Report
    print("Classification Report:")
    print(classification_report(all_labels, all_preds, target_names=class_names))
    
    # Confusion Matrix
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.title("Confusion Matrix")
    plt.ylabel("True Label")
    plt.xlabel("Predicted Label")
    plt.show()

# Uncomment after training to evaluate:
# evaluate_model(best_model, test_loader, class_names)